# theOne Market Making Strategy Backtesting Notebook

## Overview
This notebook demonstrates the usage of **theOne**, a professional market making backtesting framework that implements the Li et al. market making strategy. The framework provides a modular, extensible architecture similar to established platforms like Backtrader and Hummingbot.

## What This Notebook Does

### 1. **Framework Setup & Data Loading**
- Imports the theOne backtesting framework components:
  - `LiEtAlStrategy`: Implementation of the Li et al. market making algorithm
  - `MarketMakingBacktester`: Core backtesting engine
  - `OrderSide`: Enum for buy/sell order classification
- Loads 1-second OHLCV and order book data for POL-USDT trading pair from Binance
- Sets up the necessary Python path configurations for module imports

### 2. **Strategy Configuration**
Creates a Li et al. market making strategy with specific parameters:
- **Tick Size**: 0.0001 (minimum price increment)
- **Mu (μ)**: 2.0 (order book pressure sensitivity parameter)
- **Spread**: 0.0005 (base bid-ask spread)
- **Refresh Interval**: 5 candles (order refresh frequency)
- **Order Levels**: 3 levels of depth per side
- **Base Quantity**: 100 units (starting order size, doubles per level: 100, 200, 400)

### 3. **Portfolio Initialization**
Sets up initial portfolio conditions:
- **Base Asset**: 1000.0 POL tokens
- **Quote Asset**: $2000.0 USDT
- **Cost Basis**: $0.262 per POL token
- Matches exact conditions from the original Li et al. implementation

### 4. **Backtesting Execution**
Runs a comprehensive backtest that:
- Processes market data sequentially (1-second intervals)
- Calculates **Order Book Pressure (OBP)** using 5 candles and 5 order book levels
- Generates dynamic bid/ask quotes with OBP-based skewing
- Refreshes orders every 5 time steps
- Simulates realistic order fills using FIFO queue logic
- Tracks portfolio state, PnL, and performance metrics

### 5. **Order Management & Fill Simulation**
The strategy:
- Places 3 levels of orders on each side (bid/ask)
- Uses tick-based price adjustments and volume-weighted skewing
- Simulates fills based on market volume changes and order queue position
- Handles partial fills and order lifecycle management
- Prevents short selling (negative base balance protection)

### 6. **Performance Analysis & Visualization**
Provides comprehensive analysis through:

#### **Console Output**:
- Total fills (buy vs sell breakdown)
- Final portfolio state (base/quote balances, cost basis)
- Realized and unrealized PnL
- Total return percentage
- Fill analysis by price level and side

#### **Interactive Plotly Visualizations**:
- **PnL Evolution**: Total, realized, and unrealized PnL over time
- **Portfolio Balances**: Base and quote asset holdings with dual y-axes
- **Portfolio Value**: Total portfolio value denominated in quote currency
- **Summary Statistics Table**: Key performance metrics in tabular format

### 7. **Fill Analysis**
Detailed breakdown of trading activity:
- Fill distribution by order level (1, 2, 3)
- Price ranges and average execution prices
- Quantity analysis for buy vs sell orders
- Chronological fill tracking with timestamps

## Key Technical Features

### **Market Microstructure Modeling**
- **FIFO Queue Simulation**: Realistic order execution based on market depth
- **Volume-Based Fills**: Orders fill only when sufficient market volume exists
- **Order Book Pressure**: Dynamic pricing based on bid/ask volume imbalances

### **Portfolio Management**
- **Cost Basis Tracking**: Weighted average cost calculation for inventory
- **PnL Decomposition**: Separate tracking of realized vs unrealized gains/losses
- **Risk Management**: Prevents negative inventory positions

### **Strategy Logic**
- **Li et al. Algorithm**: Academic-grade implementation with proper OBP calculation
- **Multi-Level Orders**: Professional market maker setup with quantity scaling
- **Dynamic Skewing**: Price adjustments based on order flow imbalances

## Expected Output Patterns

When functioning correctly, the notebook should show:
- **Active Trading**: Multiple fills across different price levels
- **Balanced Activity**: Roughly similar buy and sell fill counts (market making nature)
- **Inventory Management**: Base balance fluctuating around initial 1000 tokens
- **PnL Generation**: Positive realized PnL from bid-ask spread capture
- **Reasonable Returns**: Modest positive returns consistent with market making strategies

## Use Cases for LLMs

This notebook serves as a reference for:
1. **Understanding market making strategy implementation**
2. **Learning professional backtesting framework architecture**
3. **Analyzing quantitative trading performance metrics**
4. **Comparing different market making approaches**
5. **Validating strategy behavior against academic literature**

The framework is designed to be **modular and extensible**, allowing for easy implementation of custom market making strategies while maintaining professional-grade backtesting capabilities.

In [13]:
# Cell 1: Setup
import os
import sys
import pandas as pd
import numpy as np

# Add path for theOne
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)

In [14]:
# Import theOne framework (note: no .theOne since it's a single file)
# Import theOne framework
from theOne import (
    LiEtAlStrategy, 
    MarketMakingBacktester,
    OrderSide
)
from dataHandler import load_candles_and_orderbook



In [15]:
# Cell 2: Load Data (same as original)
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
TRADING_PAIR = "POL-USDT"

candles_and_ob_df = load_candles_and_orderbook(CONNECTOR_NAME, INTERVALS, TRADING_PAIR)
print(f"Loaded {len(candles_and_ob_df)} data points")
print(f"Columns: {list(candles_and_ob_df.columns)}")

2025-06-13 15:49:28,730 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x158c05bb0>


Number of missing seconds: 0
No missing seconds. Every second has an entry.
Loaded 6027 data points
Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'n_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume', 'bids', 'asks', 'datetime', 'taker_sell_base_volume']


In [16]:
# Cell 3: Create Strategy with EXACT same parameters as original
strategy = LiEtAlStrategy(
    tick_size=0.0001,
    mu=2.0,  # CRITICAL: Use 2.0 to match original, not 3.0
    spread=0.0005,
    refresh_interval=5,
    n_candles=5,
    orderbook_levels=5,
    order_levels=3,
    base_quantity=100.0
)

print(f"Strategy created: {strategy.name}")
print(f"Parameters: mu={strategy.mu}, refresh_interval={strategy.refresh_interval}")

Strategy created: LiEtAl
Parameters: mu=2.0, refresh_interval=5


In [17]:
# Cell 4: Create Backtester with EXACT same initial conditions
backtester = MarketMakingBacktester(
    strategy=strategy,
    initial_base=1000.0,      # Same as original
    initial_quote=2000.0,     # Same as original  
    initial_cost_basis=0.262  # Same as original
)

print(f"Initial Portfolio:")
print(f"  Base: {backtester.portfolio.base_balance}")
print(f"  Quote: {backtester.portfolio.quote_balance}")
print(f"  Cost Basis: {backtester.portfolio.base_cost_basis}")

Initial Portfolio:
  Base: 1000.0
  Quote: 2000.0
  Cost Basis: 0.262


In [18]:
# Cell 5: Run Backtest
print("Starting backtest...")
results = backtester.run_backtest(candles_and_ob_df)

print("Backtest completed!")
print(f"Strategy: {results['strategy_name']}")
print(f"Total Fills: {results['total_fills']}")
print(f"  - Buy Fills: {results['buy_fills']}")
print(f"  - Sell Fills: {results['sell_fills']}")

Starting backtest...


2025-06-13 15:49:29,185 - asyncio - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-10' coro=<safe_wrapper() running at /opt/homebrew/anaconda3/envs/quants-lab/lib/python3.12/site-packages/hummingbot/core/utils/async_utils.py:9> wait_for=<Future pending cb=[Task.task_wakeup()]>>


Backtest completed!
Strategy: LiEtAl
Total Fills: 155
  - Buy Fills: 38
  - Sell Fills: 117


In [19]:
# Cell 6: Compare Results with Original
print("=== PORTFOLIO PERFORMANCE ===")
print(f"Initial Portfolio Value: ${results['initial_portfolio_value']:.2f}")
print(f"Final Portfolio Value: ${results['final_portfolio_value']:.2f}")
print(f"Total Return: {results['total_return_pct']:.2f}%")
print()
print(f"Final Realized PnL: ${results['final_realized_pnl']:.2f}")
print(f"Final Unrealized PnL: ${results['final_unrealized_pnl']:.2f}")
print(f"Final Total PnL: ${results['final_total_pnl']:.2f}")
print()
print(f"Final Portfolio State:")
print(f"  Base Balance: {results['portfolio'].base_balance:.2f}")
print(f"  Quote Balance: ${results['portfolio'].quote_balance:.2f}")
print(f"  Cost Basis: ${results['portfolio'].base_cost_basis:.4f}")

=== PORTFOLIO PERFORMANCE ===
Initial Portfolio Value: $2211.20
Final Portfolio Value: $3635.01
Total Return: 64.39%

Final Realized PnL: $-53.09
Final Unrealized PnL: $0.09
Final Total PnL: $-53.01

Final Portfolio State:
  Base Balance: 175.90
  Quote Balance: $3597.67
  Cost Basis: $0.2118


In [20]:
# Cell 8: Detailed Fill Analysis
fills_df = pd.DataFrame([{
    'timestamp': f.timestamp,
    'side': f.side.value,
    'price': f.price,
    'quantity': f.quantity,
    'level': f.level
} for f in results['fills']])

if len(fills_df) > 0:
    print("=== FILL ANALYSIS ===")
    print(f"Total Fills: {len(fills_df)}")
    
    # Group by side
    buy_fills = fills_df[fills_df['side'] == 'buy']
    sell_fills = fills_df[fills_df['side'] == 'sell']
    
    print(f"Buy Fills: {len(buy_fills)}")
    if len(buy_fills) > 0:
        print(f"  - Total Quantity: {buy_fills['quantity'].sum():.2f}")
        print(f"  - Avg Price: ${buy_fills['price'].mean():.4f}")
        print(f"  - Price Range: ${buy_fills['price'].min():.4f} - ${buy_fills['price'].max():.4f}")
    
    print(f"Sell Fills: {len(sell_fills)}")
    if len(sell_fills) > 0:
        print(f"  - Total Quantity: {sell_fills['quantity'].sum():.2f}")
        print(f"  - Avg Price: ${sell_fills['price'].mean():.4f}")
        print(f"  - Price Range: ${sell_fills['price'].min():.4f} - ${sell_fills['price'].max():.4f}")
    
    # Fill distribution by level
    print("\nFills by Level:")
    level_counts = fills_df.groupby(['side', 'level']).size().unstack(fill_value=0)
    print(level_counts)
else:
    print("No fills occurred during backtest")

=== FILL ANALYSIS ===
Total Fills: 155
Buy Fills: 38
  - Total Quantity: 3766.80
  - Avg Price: $0.2116
  - Price Range: $0.2110 - $0.2122
Sell Fills: 117
  - Total Quantity: 11309.90
  - Avg Price: $0.2117
  - Price Range: $0.2112 - $0.2123

Fills by Level:
level   1   2  3
side            
buy    28  10  0
sell   88  28  1


In [25]:
# Cell 7: Analyze Performance Time Series with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

performance_df = results['performance_timeseries']

# Create subplots with secondary y-axis for the middle plot
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('PnL Over Time', 'Portfolio Balances', 'Total Portfolio Value Over Time'),
    specs=[[{"secondary_y": False}],
           [{"secondary_y": True}],
           [{"secondary_y": False}]],
    vertical_spacing=0.08
)

# Plot 1: PnL Components
fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['total_pnl'],
        name='Total PnL',
        line=dict(color='blue', width=2),
        hovertemplate='<b>Total PnL</b><br>Time: %{x}<br>Value: $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['realized_pnl'],
        name='Realized PnL',
        line=dict(color='green', width=2),
        hovertemplate='<b>Realized PnL</b><br>Time: %{x}<br>Value: $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['unrealized_pnl'],
        name='Unrealized PnL',
        line=dict(color='orange', width=2),
        hovertemplate='<b>Unrealized PnL</b><br>Time: %{x}<br>Value: $%{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Plot 2: Portfolio Balances (with secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['base_balance'],
        name='Base Balance',
        line=dict(color='red', width=2),
        hovertemplate='<b>Base Balance</b><br>Time: %{x}<br>Balance: %{y:.4f}<extra></extra>'
    ),
    row=2, col=1, secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['quote_balance'],
        name='Quote Balance',
        line=dict(color='blue', width=2),
        hovertemplate='<b>Quote Balance</b><br>Time: %{x}<br>Balance: $%{y:.2f}<extra></extra>'
    ),
    row=2, col=1, secondary_y=True
)

# Plot 3: Total Portfolio Value
fig.add_trace(
    go.Scatter(
        x=performance_df['timestamp'], 
        y=performance_df['total_value'],
        name='Total Portfolio Value',
        line=dict(color='purple', width=2),
        fill='tonexty',
        fillcolor='rgba(128, 0, 128, 0.1)',
        hovertemplate='<b>Portfolio Value</b><br>Time: %{x}<br>Value: $%{y:.2f}<extra></extra>'
    ),
    row=3, col=1
)

# Update layout and axis labels
fig.update_layout(
    height=900,
    title_text="Market Making Strategy Performance Dashboard",
    title_x=0.5,
    title_font_size=20,
    showlegend=True,
    hovermode='x unified',
    template='plotly_white'
)
result = TRADING_PAIR.split("-", 1)
# Update y-axis labels
fig.update_yaxes(title_text="PnL ($)", row=1, col=1)
fig.update_yaxes(title_text=f"Base Balance {result[0]}", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text=f"Quote Balance {result[1]}", row=2, col=1, secondary_y=True)
fig.update_yaxes(title_text="Portfolio Value ($)", row=3, col=1)

# Update x-axis labels
fig.update_xaxes(title_text="Timestamp", row=3, col=1)

# Color the secondary y-axis labels to match the traces
fig.update_yaxes(title_font_color="red", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_font_color="blue", row=2, col=1, secondary_y=True)

# Show the interactive plot
fig.show()

In [22]:
# Bonus: Interactive Summary Table
summary_data = {
    'Metric': [
        'Initial Portfolio Value',
        'Final Portfolio Value', 
        'Total Return (%)',
        'Total PnL',
        'Realized PnL',
        'Unrealized PnL',
        'Total Fills',
        'Buy Fills',
        'Sell Fills',
        'Final Base Balance',
        'Final Quote Balance',
        'Average Cost Basis'
    ],
    'Value': [
        f"${results['initial_portfolio_value']:.2f}",
        f"${results['final_portfolio_value']:.2f}",
        f"{results['total_return_pct']:.2f}%",
        f"${results['final_total_pnl']:.2f}",
        f"${results['final_realized_pnl']:.2f}",
        f"${results['final_unrealized_pnl']:.2f}",
        f"{results['total_fills']}",
        f"{results['buy_fills']}",
        f"{results['sell_fills']}",
        f"{results['portfolio'].base_balance:.4f}",
        f"${results['portfolio'].quote_balance:.2f}",
        f"${results['portfolio'].base_cost_basis:.4f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Performance Metric</b>', '<b>Value</b>'],
        fill_color='lightblue',
        font_size=14,
        height=30
    ),
    cells=dict(
        values=[summary_df['Metric'], summary_df['Value']],
        fill_color='white',
        font_size=12,
        height=25,
        align='left'
    )
)])

fig_table.update_layout(
    title="📋 theOne Backtest Summary Statistics",
    title_x=0.5,
    title_font_size=18,
    height=400
)

fig_table.show()